In [1]:
import torch
import numpy as np

def salvar_dataset_npz(loader, arquivo):
    dados = []
    labels = []

    for batch in loader:
        # formato (x, y)
        x, y = batch

        # PyTorch
        if torch.is_tensor(x):
            x = x.cpu().numpy()
            y = y.cpu().numpy()

        dados.append(x)
        labels.append(y)

    dados = np.concatenate(dados, axis=0)
    labels = np.concatenate(labels, axis=0)

    np.savez_compressed(
        arquivo,
        data=dados,
        labels=labels
    )

    print(f"Arquivo '{arquivo}.npz' salvo.")

# Testes Tonic

In [ ]:
%pip install tonic
import tonic

In [ ]:
import tonic.transforms as transforms

num_steps = 10

sensor_size = tonic.datasets.NMNIST.sensor_size

transform = transforms.Compose(
    [
        transforms.ToFrame(sensor_size=sensor_size, n_time_bins=num_steps),
    ]
)


test_set = tonic.datasets.NMNIST(save_to="data/nmnist", train=False, transform=transform)

169675776it [01:13, 2316677.23it/s]                               


Extracting data/nmnist/NMNIST/test.zip to data/nmnist/NMNIST


In [15]:
data, label = test_set[1000]

In [16]:
print(data.shape)

(10, 2, 34, 34)


In [17]:
print(label)

1


# Criando Dados Pequenos

In [2]:
import numpy as np

X_np = np.array([
    # amostra 0
    [
        [1,0,0,0,1],
        [0,1,0,0,0],
        [0,0,1,0,0],
    ],
    # amostra 1
    [
        [0,1,1,0,0],
        [0,0,1,1,0],
        [0,0,0,1,1],
    ],
    # amostra 2
    [
        [1,0,0,0,0],
        [1,0,0,0,0],
        [0,1,0,0,0],
    ],
    # amostra 3
    [
        [0,0,0,0,1],
        [0,0,0,1,0],
        [0,0,1,0,0],
    ],
], dtype=np.float32)

Y_np = np.array([0, 1, 2, 3], dtype=np.float32)

## Torch

In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.from_numpy(X_np)
Y = torch.from_numpy(Y_np)

dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

In [4]:
salvar_dataset_npz(dataloader, "z_mini_torch.npz")

Arquivo 'z_mini_torch.npz.npz' salvo.


## Tensorflow

In [5]:
import tensorflow as tf
import numpy as np

batch_size = 1

# Cria o Dataset
dataset = tf.data.Dataset.from_tensor_slices((X_np, Y_np))

# Embaralha e cria batches
dataset = dataset.batch(batch_size)

# Teste
for xb, yb in dataset.take(1):
    print(xb.shape, yb.shape)

2026-01-13 16:11:12.274925: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-13 16:11:12.304073: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 16:11:12.304113: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 16:11:12.304119: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 16:11:12.309100: I tensorflow/core/platform/cpu_feature_g

(1, 3, 5) (1,)


2026-01-13 16:11:13.721023: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-01-13 16:11:13.742342: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-01-13 16:11:13.743772: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [6]:
salvar_dataset_npz(dataset, "z_mini_tf.npz")

Arquivo 'z_mini_tf.npz.npz' salvo.


# NeuroHLS

In [7]:
from NeuroHls import *

In [8]:
neuro_hls = NeuroHls("z_test")

In [9]:
model_config = neuro_hls.get_dummy_model_config()

In [10]:
print(model_config)

------------------------------
Layer 1: Dense (784, 128)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8

------------------------------
Layer 2: Dense (128, 10)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8



In [11]:
neuro_hls.implement_model_from_config(model_config)

## Criando o Testbench

In [ ]:
neuro_hls.define_test_dataset("z_mini_torch.npz", data_is_binary=True, step_count=10, different_sample_per_step=False)

In [ ]:
neuro_hls.create_testbench(total_samples=100, batch_size=30)

Testbench Criado
